In [189]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [190]:
df=pd.read_csv("/content/mail_data.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [191]:
df["Message"]

,Message
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
...,...
5567,This is the 2nd time we have tried 2 contact u...
5568,Will ü b going to esplanade fr home?
5569,"Pity, * was in mood for that. So...any other s..."
5570,The guy did some bitching but I acted like i'd...


In [192]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [193]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.stem import wordnet, WordNetLemmatizer
from nltk.corpus import stopwords

In [194]:
import re

In [195]:
def data_converting(data):
  data=data.lower()
  data=re.sub('[^A-Za-z0-9]',' ',data)
  data=data.split()
  data= [WordNetLemmatizer().lemmatize(word) for word in data if not word in stopwords.words('english')]
  data=" ".join(data)
  return data


In [196]:
print(data_converting("Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."))

go jurong point crazy available bugis n great world la e buffet cine got amore wat


In [197]:
df["Message"]=df["Message"].apply(data_converting)

In [198]:
df.head()

,Category,Message
0,ham,go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni
2,spam,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,u dun say early hor u c already say
4,ham,nah think go usf life around though


## Create Bag of words

In [ ]:
## another Method 
df["Category"]=df["Category"].map({"spam":1,"ham":0})
df.head()

## or 
df["Category"]=df["Category"].apply(lambda x:1 if x=="spam" else 0)
df.head()

## another method
df.loc[df["Category"]=="spam","Category"]=1
df.loc[df["Category"]=="ham","Category"]=0
df.head()

## another method
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["Category"]=le.fit_transform(df["Category"])
df.head()

## another method
df["Category"]=df["Category"].replace({"spam":1,"ham":0})
df.head()

## another method
df["Category"]=df["Category"].astype(int)
df.head()

## another method
df["Category"]=df["Category"].apply(lambda x:int(x))
df.head()

In [199]:
y=pd.get_dummies(df["Category"])
y=y.iloc[:,0].values

In [200]:
y

array([ True,  True, False, ...,  True,  True,  True])

In [201]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df["Message"],y,test_size=0.20)

In [202]:
# ## Create the Bag OF Words model
# from sklearn.feature_extraction.text import CountVectorizer
# ## for Binary BOW enable binary=True
# cv=CountVectorizer(max_features=2500,ngram_range=(1,2))

In [203]:
len(X_train),len(y_train)

(4457, 4457)

In [204]:
# ## independent features
# X_train=cv.fit_transform(X_train).toarray()
# X_test=cv.transform(X_test).toarray()

In [205]:
# cv.vocabulary_

In [206]:
# from sklearn.naive_bayes import MultinomialNB
# spam_detect_model=MultinomialNB().fit(X_train,y_train)
# y_pred=spam_detect_model.predict(X_test)
# from sklearn.metrics import accuracy_score,classification_report
# accuracy_score(y_test,y_pred)

In [207]:
# from sklearn.metrics import classification_report
# print(classification_report(y_test,y_pred))

## TFIDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
tfid= TfidfVectorizer(max_features=2500,ngram_range=(1,2))
# tfid = TfidfVectorizer(min_df=1,stop_words="english",lowercase="True")
## here min_df ==> it says that if the words are occuring only once we can ignore it or assign very low frequency points
## stop_words ===> we can remove the stop words like (this, the , untill....etc.)
## lowercase ---> making sure all the charcters in lowercase
X_train_tf = tfid.fit_transform(X_train).toarray()
X_test_tf= tfid.transform(X_test).toarray()
print(len(X_train_tf),len(X_test_tf))
## Model training
model_log= LogisticRegression()
model_log.fit(X_train_tf,y_train)
y_pred_log=model_log.predict(X_test_tf)

## Metrics
print(accuracy_score(y_test,y_pred_log))
print(classification_report(y_test,y_pred_log))

4457 1115
0.9775784753363229
              precision    recall  f1-score   support

       False       0.98      0.84      0.91       148
        True       0.98      1.00      0.99       967

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [210]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
tfid= TfidfVectorizer(max_features=2500,ngram_range=(1,2))
X_train = tfid.fit_transform(X_train).toarray()
X_test=tfid.transform(X_test).toarray()
print(len(X_train),len(y_train))
## Model training
model_log= MultinomialNB()
model_log.fit(X_train,y_train)
y_pred_log=model_log.predict(X_test)

## Metrics
print(accuracy_score(y_test,y_pred_log))
print(classification_report(y_test,y_pred_log))

4457 4457
0.9802690582959641
              precision    recall  f1-score   support

       False       0.98      0.86      0.92       148
        True       0.98      1.00      0.99       967

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [213]:
prediction="Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"

pred=model_log.predict(tfid.transform([prediction]))
print(pred)

if pred[0]==0:
  print("spam")
else:
  print("Ham")

[False]
spam
